# UD2.05. Voz y traducción con Azure AI Speech y Translator

**Módulo 5073 · Programación de Inteligencia Artificial · UD2 · Práctica P2.2**

Speech no comparte endpoint con el resto del catálogo. Tiene **un host por región y otro por
operación**, y eso confunde al llegar de Language y Vision, donde todo salía del mismo sitio.

| Operación | Host |
|---|---|
| Voz a texto (audio corto) | `https://{region}.stt.speech.microsoft.com/...` |
| Texto a voz | `https://{region}.tts.speech.microsoft.com/...` |
| Traducción de texto | `https://api.cognitive.microsofttranslator.com/translate` |

Translator, además, es global: no lleva región en el host, pero sí una cabecera con la región del
recurso.

Este cuaderno construye `servicios/voz.py` de la práctica P2.2.

> **La voz es un dato biométrico.** Una grabación permite identificar a quien habla, y el RGPD le
> da protección reforzada. En este cuaderno se usan **grabaciones propias**. No subas al servicio
> audios de otras personas sin su consentimiento explícito.

In [ ]:
!pip install -q requests python-dotenv

In [ ]:
import os
import pathlib
import wave
from xml.sax.saxutils import escape

import requests
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

CLAVE_VOZ = os.getenv("AZURE_SPEECH_KEY") or os.getenv("AZURE_LANGUAGE_KEY")
REGION = os.getenv("AZURE_REGION", "westeurope")
CLAVE_TRADUCTOR = os.getenv("AZURE_TRANSLATOR_KEY") or CLAVE_VOZ

print("Clave de voz cargada:", bool(CLAVE_VOZ))
print("Región:              ", REGION)

if not CLAVE_VOZ:
    print("\nFalta configuración. Repasa el cuaderno UD2.02 antes de seguir.")

La **región** aquí no es un detalle de configuración: forma parte de la URL. Si tu recurso está
en `westeurope` y llamas a `francecentral`, la respuesta es un 401, no un 404, porque la clave no
es válida en esa región. Es uno de los fallos que más cuesta diagnosticar.

Y la región también decide **dónde se procesa el audio**, que es lo que importa para el RGPD.

## 1. Comprobar el audio antes de enviarlo

El endpoint de audio corto acepta WAV PCM de 16 bits, **16 kHz** y **mono**, y hasta unos 60
segundos. El `Content-Type` de la petición declara ese formato:

```
Content-Type: audio/wav; codecs=audio/pcm; samplerate=16000
```

Y aquí está la trampa de esta parte: **un WAV a 44,1 kHz declarado como 16 kHz no da error**. El
servicio lo interpreta a la frecuencia que le has dicho, y devuelve una transcripción incoherente
o vacía. Un fallo silencioso es peor que una excepción, así que hay que comprobarlo en local.

El módulo `wave` de la biblioteca estándar basta.

In [ ]:
FRECUENCIA = 16000
CANALES = 1
ANCHO_MUESTRA = 2          # 16 bits
DURACION_MAXIMA = 60


class AudioNoValido(ValueError):
    """El audio no cumple lo que espera el endpoint de audio corto."""


def inspecciona_wav(ruta) -> dict:
    """Devuelve las características del WAV, o explica por qué no sirve."""
    try:
        with wave.open(str(ruta), "rb") as f:
            info = {
                "canales": f.getnchannels(),
                "frecuencia": f.getframerate(),
                "ancho_muestra": f.getsampwidth(),
                "segundos": f.getnframes() / f.getframerate(),
            }
    except wave.Error as error:
        raise AudioNoValido(f"No es un WAV legible: {error}") from error

    problemas = []
    if info["canales"] != CANALES:
        problemas.append(f"tiene {info['canales']} canales y hace falta mono")
    if info["frecuencia"] != FRECUENCIA:
        problemas.append(f"va a {info['frecuencia']} Hz y hace falta {FRECUENCIA} Hz")
    if info["ancho_muestra"] != ANCHO_MUESTRA:
        problemas.append(f"usa {info['ancho_muestra'] * 8} bits por muestra y hacen falta 16")
    if info["segundos"] > DURACION_MAXIMA:
        problemas.append(f"dura {info['segundos']:.0f} s y el máximo son {DURACION_MAXIMA} s")

    info["valido"] = not problemas
    info["problemas"] = problemas
    return info

Y si no cumple, se convierte. `ffmpeg` está en Colab y se instala en cualquier sistema:

```bash
ffmpeg -i entrada.m4a -ac 1 -ar 16000 -sample_fmt s16 salida.wav
```

- `-ac 1` → un canal, mono
- `-ar 16000` → frecuencia de muestreo de 16 kHz
- `-sample_fmt s16` → 16 bits por muestra

In [ ]:
# Comprobación sin servicio: generamos dos WAV, uno correcto y uno que no lo es
import math
import struct

def genera_wav(ruta, frecuencia=16000, canales=1, segundos=1.0):
    """Genera un tono de prueba con las características indicadas."""
    with wave.open(str(ruta), "wb") as f:
        f.setnchannels(canales)
        f.setsampwidth(2)
        f.setframerate(frecuencia)
        for i in range(int(frecuencia * segundos)):
            muestra = int(16000 * math.sin(2 * math.pi * 440 * i / frecuencia))
            f.writeframes(struct.pack("<h", muestra) * canales)


genera_wav("prueba_correcta.wav", frecuencia=16000, canales=1)
genera_wav("prueba_mala.wav", frecuencia=44100, canales=2)

for nombre in ("prueba_correcta.wav", "prueba_mala.wav"):
    info = inspecciona_wav(nombre)
    estado = "válido" if info["valido"] else "NO válido: " + "; ".join(info["problemas"])
    print(f"{nombre:22} {info['frecuencia']} Hz, {info['canales']} canal(es) -> {estado}")

## 2. Voz a texto

In [ ]:
class ErrorServicio(Exception):
    """Fallo al hablar con el servicio."""


def transcribe(ruta_wav, idioma="es-ES", timeout=60) -> dict:
    """Transcribe un WAV corto. Devuelve el texto reconocido y la confianza."""
    info = inspecciona_wav(ruta_wav)
    if not info["valido"]:
        raise AudioNoValido(
            "El audio no cumple lo que espera el servicio: "
            + "; ".join(info["problemas"])
            + ". Conviértelo con: ffmpeg -i entrada -ac 1 -ar 16000 -sample_fmt s16 salida.wav"
        )

    url = (f"https://{REGION}.stt.speech.microsoft.com"
           "/speech/recognition/conversation/cognitiveservices/v1")
    cabeceras = {
        "Ocp-Apim-Subscription-Key": CLAVE_VOZ or "",
        "Content-Type": "audio/wav; codecs=audio/pcm; samplerate=16000",
        "Accept": "application/json",
    }
    parametros = {"language": idioma, "format": "detailed"}

    try:
        respuesta = requests.post(url, headers=cabeceras, params=parametros,
                                  data=pathlib.Path(ruta_wav).read_bytes(), timeout=timeout)
    except requests.RequestException as error:
        raise ErrorServicio(f"No se pudo contactar: {error}") from error

    if respuesta.status_code in (401, 403):
        raise ErrorServicio(
            f"Credenciales rechazadas (HTTP {respuesta.status_code}). "
            f"Comprueba que la clave corresponde a la región {REGION}."
        )
    if respuesta.status_code != 200:
        raise ErrorServicio(f"HTTP {respuesta.status_code}: {respuesta.text[:300]}")

    datos = respuesta.json()
    if datos.get("RecognitionStatus") != "Success":
        return {"texto": "", "estado": datos.get("RecognitionStatus"), "confianza": None,
                "duracion_s": info["segundos"]}

    mejor = (datos.get("NBest") or [{}])[0]
    return {
        "texto": datos.get("DisplayText") or mejor.get("Display", ""),
        "estado": "Success",
        "confianza": mejor.get("Confidence"),
        "duracion_s": info["segundos"],
    }

Dos cosas que no son evidentes:

- **`format=detailed`** hace que la respuesta traiga `NBest`, con varias hipótesis y su
  confianza. Sin él solo llega el texto, y la confianza es justo lo que hay que enseñar en la
  interfaz.
- **`RecognitionStatus` distinto de `Success` no es un error HTTP.** Un audio en silencio
  devuelve 200 con estado `NoMatch`. Si no lo compruebas, tu aplicación enseña una transcripción
  vacía sin explicar por qué.

In [ ]:
RUTA_AUDIO = pathlib.Path("voz_es.wav")

if CLAVE_VOZ and RUTA_AUDIO.exists():
    resultado = transcribe(RUTA_AUDIO)
    print(f"Estado:    {resultado['estado']}")
    print(f"Confianza: {resultado['confianza']}")
    print(f"Texto:     {resultado['texto']}")
else:
    print(f"Graba una frase tuya en {RUTA_AUDIO} (WAV PCM 16 kHz mono) para ejecutar esta celda")

## 3. Texto a voz, con SSML

La síntesis no se pide con texto plano: se pide con **SSML**, un lenguaje de marcado en XML que
describe cómo hay que leer el texto. Pausas, énfasis, velocidad, tono, pronunciación de siglas.

```xml
<speak version='1.0' xml:lang='es-ES'>
  <voice xml:lang='es-ES' name='es-ES-ElviraNeural'>
    Hola. <break time='400ms'/> Esta frase lleva una pausa,
    y <emphasis level='strong'>esta palabra</emphasis> va enfatizada.
  </voice>
</speak>
```

> Aquí vuelve el criterio 1.f de la UD1. SSML es exactamente lo mismo que viste con XML:
> **etiquetas que describen cómo interpretar el contenido**, no qué hacer con él. `<break>` no
> ejecuta nada; declara que ahí hay una pausa, y el motor decide cómo la realiza.
>
> Para la práctica hay que explicar qué aporta cada etiqueta que uses. Esto es marcado, y se
> analiza como marcado.

In [ ]:
VOCES = {
    "Elvira (es-ES, femenina)": "es-ES-ElviraNeural",
    "Álvaro (es-ES, masculina)": "es-ES-AlvaroNeural",
    "Dalia (es-MX, femenina)": "es-MX-DaliaNeural",
}

FORMATOS_SALIDA = {
    "mp3": "audio-16khz-128kbitrate-mono-mp3",
    "wav": "riff-16khz-16bit-mono-pcm",
}


def construye_ssml(texto, voz="es-ES-ElviraNeural", idioma="es-ES",
                   velocidad=None, pausa_final_ms=None) -> str:
    """Construye el SSML escapando el texto del usuario."""
    contenido = escape(texto)               # el texto puede traer <, > o &
    if velocidad:
        contenido = f"<prosody rate='{velocidad}'>{contenido}</prosody>"
    if pausa_final_ms:
        contenido += f"<break time='{int(pausa_final_ms)}ms'/>"
    return (
        f"<speak version='1.0' xml:lang='{idioma}'>"
        f"<voice xml:lang='{idioma}' name='{voz}'>{contenido}</voice>"
        f"</speak>"
    )


print(construye_ssml("Hola. Esto es una prueba con <acentos> y & símbolos.",
                     velocidad="0.9", pausa_final_ms=500))

El `escape()` no es cosmético. Si el texto que teclea el usuario lleva un `<` y lo metes tal cual
en el SSML, el XML deja de estar bien formado y el servicio responde 400. Peor: un texto
cuidadosamente construido podría inyectar etiquetas SSML propias. Es el mismo problema de siempre
al construir marcado concatenando cadenas.

In [ ]:
def sintetiza(texto, voz="es-ES-ElviraNeural", formato="mp3", timeout=60, **ssml) -> bytes:
    """Devuelve los bytes del audio generado."""
    url = f"https://{REGION}.tts.speech.microsoft.com/cognitiveservices/v1"
    cabeceras = {
        "Ocp-Apim-Subscription-Key": CLAVE_VOZ or "",
        "Content-Type": "application/ssml+xml",
        "X-Microsoft-OutputFormat": FORMATOS_SALIDA[formato],
        "User-Agent": "modulo-5073-ud2",
    }
    cuerpo = construye_ssml(texto, voz=voz, **ssml).encode("utf-8")

    try:
        respuesta = requests.post(url, headers=cabeceras, data=cuerpo, timeout=timeout)
    except requests.RequestException as error:
        raise ErrorServicio(f"No se pudo contactar: {error}") from error

    if respuesta.status_code == 200:
        return respuesta.content
    if respuesta.status_code == 400:
        raise ErrorServicio(f"SSML incorrecto o voz inexistente: {respuesta.text[:300]}")
    if respuesta.status_code in (401, 403):
        raise ErrorServicio(f"Credenciales rechazadas para la región {REGION}")
    raise ErrorServicio(f"HTTP {respuesta.status_code}: {respuesta.text[:300]}")

`User-Agent` es obligatorio en el endpoint de síntesis: sin él, algunas regiones responden 400 sin
explicar el motivo. Es el tipo de requisito que solo está en la documentación y que no se deduce
del error.

In [ ]:
if CLAVE_VOZ:
    audio = sintetiza(
        "Bienvenida al módulo de programación de inteligencia artificial. "
        "Esta voz está generada por un servicio en la nube.",
        voz="es-ES-ElviraNeural", velocidad="0.95", pausa_final_ms=300,
    )
    pathlib.Path("salida_tts.mp3").write_bytes(audio)
    print(f"Generados {len(audio)} bytes en salida_tts.mp3")

    from IPython.display import Audio
    display(Audio("salida_tts.mp3"))

## 4. Traducción

Translator es distinto de los dos anteriores en tres cosas: host global, la región va en una
**cabecera**, y el cuerpo es una **lista** de objetos, no un objeto con documentos dentro.

Cada servicio tiene su forma. Por eso el módulo `servicios/` existe: para que el resto del
programa no tenga que saber esto.

In [ ]:
def traduce(textos, origen=None, destinos=("en",), timeout=30) -> list[dict]:
    """Traduce una lista de textos a uno o varios idiomas destino."""
    url = "https://api.cognitive.microsofttranslator.com/translate"
    parametros = {"api-version": "3.0", "to": list(destinos)}
    if origen:
        parametros["from"] = origen         # si no se indica, lo detecta el servicio

    cabeceras = {
        "Ocp-Apim-Subscription-Key": CLAVE_TRADUCTOR or "",
        "Ocp-Apim-Subscription-Region": REGION,
        "Content-Type": "application/json",
    }
    cuerpo = [{"text": t} for t in textos]

    try:
        respuesta = requests.post(url, headers=cabeceras, params=parametros,
                                  json=cuerpo, timeout=timeout)
    except requests.RequestException as error:
        raise ErrorServicio(f"No se pudo contactar: {error}") from error

    if respuesta.status_code != 200:
        raise ErrorServicio(f"HTTP {respuesta.status_code}: {respuesta.text[:300]}")

    salida = []
    for item in respuesta.json():
        salida.append({
            "detectado": (item.get("detectedLanguage") or {}).get("language"),
            "traducciones": {t["to"]: t["text"] for t in item["translations"]},
        })
    return salida

In [ ]:
if CLAVE_TRADUCTOR:
    for r in traduce(["Bon dia, voldria demanar cita.",
                      "El plazo de presentación termina el viernes."],
                     destinos=("en", "fr")):
        print(r)

Prueba a traducir una frase en valenciano. Translator lo trata como catalán (`ca`), igual que
Language: es la etiqueta que existe en el estándar. Anótalo, porque es material para la
comparativa de A2.1.

## 5. Encadenar los tres: traductor por voz

```
voz  →  texto  →  traducción  →  voz
STT      Translator                TTS
```

Este es el proyecto propuesto número 1 de PR2, y aquí está el flujo entero en una función.

In [ ]:
def traductor_por_voz(ruta_wav, idioma_origen="es-ES", destino="en",
                      voz_destino="en-US-JennyNeural") -> dict:
    """Transcribe, traduce y sintetiza. Devuelve los pasos intermedios y el audio."""
    transcripcion = transcribe(ruta_wav, idioma=idioma_origen)
    if not transcripcion["texto"]:
        raise ErrorServicio(
            f"No se ha reconocido nada en el audio (estado {transcripcion['estado']})"
        )

    traduccion = traduce([transcripcion["texto"]],
                         origen=idioma_origen.split("-")[0],
                         destinos=(destino,))[0]["traducciones"][destino]

    audio = sintetiza(traduccion, voz=voz_destino)

    return {
        "original": transcripcion["texto"],
        "confianza_stt": transcripcion["confianza"],
        "traduccion": traduccion,
        "audio": audio,
    }

In [ ]:
if CLAVE_VOZ and RUTA_AUDIO.exists():
    r = traductor_por_voz(RUTA_AUDIO)
    print("Original:  ", r["original"])
    print("Traducción:", r["traduccion"])
    pathlib.Path("traducido.mp3").write_bytes(r["audio"])

    from IPython.display import Audio
    display(Audio("traducido.mp3"))

**Enseña los pasos intermedios, no solo el resultado.** Un error de transcripción se arrastra a la
traducción, y quien usa la aplicación no puede entender qué ha pasado si solo ve el audio final.
Es un requisito del proyecto 1 de PR2, y es una decisión de diseño, no un detalle.

Para la práctica hay que comentar **en qué paso se pierde más calidad**. La respuesta habitual es
el primero: si la transcripción falla, todo lo demás es correcto sobre una entrada equivocada.

## 6. Lo que cuesta

Las unidades de facturación son distintas en cada servicio, y por eso el cálculo no se puede
copiar de uno a otro:

| Servicio | Unidad de facturación |
|---|---|
| Voz a texto | Por **hora de audio** procesada |
| Texto a voz | Por **millón de caracteres** sintetizados |
| Translator | Por **millón de caracteres** traducidos |

Consulta los precios vigentes y anota la fecha: son el dato que más rápido se queda viejo de todo
el material de esta unidad.

Y en un traductor por voz, ten en cuenta que **una interacción son tres llamadas** a tres
servicios con tres tarifas distintas.

In [ ]:
# Limpieza de los ficheros de prueba generados por este cuaderno
for nombre in ("prueba_correcta.wav", "prueba_mala.wav"):
    pathlib.Path(nombre).unlink(missing_ok=True)
print("Ficheros de prueba eliminados")

## Lo que te llevas a `servicios/voz.py`

`inspecciona_wav`, `transcribe`, `construye_ssml`, `sintetiza` y `traduce`. `traductor_por_voz`
no: eso ya es lógica de aplicación y va en la página de Streamlit que la usa.

Siguiente: **UD2.06**, donde todo esto se convierte en una aplicación.